# Data Cleaning & Transformation

# Import libraries

In [26]:
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

# Load the CSV

In [4]:
df = pd.read_csv("Hospital ER_Data.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (9216, 12)


,Patient Id,Patient Admission Date,Patient First Inital,Patient Last Name,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime,Patients CM
0,145-39-5406,20-03-2024 08:47,H,Glasspool,M,69,White,NaN,False,10.0,39,0
1,316-34-3057,15-06-2024 11:29,X,Methuen,M,4,Native American/Alaska Native,NaN,True,NaN,27,0
2,897-46-3852,20-06-2024 09:13,P,Schubuser,F,56,African American,General Practice,True,9.0,55,0
3,358-31-9711,04-02-2024 22:34,U,Titcombe,F,24,Native American/Alaska Native,General Practice,True,8.0,31,0
4,289-26-0537,04-09-2024 17:48,Y,Gionettitti,M,5,African American,Orthopedics,False,NaN,10,0


# Basic information

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9216 entries, 0 to 9215
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Patient Id                  9216 non-null   str    
 1   Patient Admission Date      9216 non-null   str    
 2   Patient First Inital        9216 non-null   str    
 3   Patient Last Name           9216 non-null   str    
 4   Patient Gender              9216 non-null   str    
 5   Patient Age                 9216 non-null   int64  
 6   Patient Race                9216 non-null   str    
 7   Department Referral         3816 non-null   str    
 8   Patient Admission Flag      9216 non-null   bool   
 9   Patient Satisfaction Score  2517 non-null   float64
 10  Patient Waittime            9216 non-null   int64  
 11  Patients CM                 9216 non-null   int64  
dtypes: bool(1), float64(1), int64(3), str(7)
memory usage: 801.1 KB


In [6]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 9216
Columns: 12


# Check column names

In [7]:
print(df.columns.tolist())

['Patient Id', 'Patient Admission Date', 'Patient First Inital', 'Patient Last Name', 'Patient Gender', 'Patient Age', 'Patient Race', 'Department Referral', 'Patient Admission Flag', 'Patient Satisfaction Score', 'Patient Waittime', 'Patients CM']


# Rename columns

In [8]:
df.rename(columns={
    "Patient First Inital": "Patient First Initial",
    "Patients CM": "Patient CM"
}, inplace=True)

print(df.columns.tolist())

['Patient Id', 'Patient Admission Date', 'Patient First Initial', 'Patient Last Name', 'Patient Gender', 'Patient Age', 'Patient Race', 'Department Referral', 'Patient Admission Flag', 'Patient Satisfaction Score', 'Patient Waittime', 'Patient CM']


# Check data types

In [9]:
df.dtypes

Patient Id                        str
Patient Admission Date            str
Patient First Initial             str
Patient Last Name                 str
Patient Gender                    str
Patient Age                     int64
Patient Race                      str
Department Referral               str
Patient Admission Flag           bool
Patient Satisfaction Score    float64
Patient Waittime                int64
Patient CM                      int64
dtype: object

# Convert admission date

In [10]:
df["Patient Admission Date"] = pd.to_datetime(
    df["Patient Admission Date"],
    dayfirst=True
)

In [11]:
df["Patient Admission Date"].dtype

dtype('<M8[us]')

# Check duplicates

In [12]:
duplicate_count = df.duplicated().sum()

print("Duplicate records:", duplicate_count)

Duplicate records: 0


# Check missing values

In [13]:
missing_count = df.isnull().sum()

print(missing_count)

Patient Id                       0
Patient Admission Date           0
Patient First Initial            0
Patient Last Name                0
Patient Gender                   0
Patient Age                      0
Patient Race                     0
Department Referral           5400
Patient Admission Flag           0
Patient Satisfaction Score    6699
Patient Waittime                 0
Patient CM                       0
dtype: int64


In [14]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage.round(2))

Patient Id                     0.00
Patient Admission Date         0.00
Patient First Initial          0.00
Patient Last Name              0.00
Patient Gender                 0.00
Patient Age                    0.00
Patient Race                   0.00
Department Referral           58.59
Patient Admission Flag         0.00
Patient Satisfaction Score    72.69
Patient Waittime               0.00
Patient CM                     0.00
dtype: float64


# Check rows where important patient information is completely missing

In [15]:
incomplete_records = df[
    df[["Patient Satisfaction Score", "Department Referral"]].isnull().all(axis=1)
]

print("Records with both values missing:", len(incomplete_records))

Records with both values missing: 3960


In [16]:
df = df.dropna(
    subset=["Patient Satisfaction Score", "Department Referral"],
    how="all"
)

print("Shape after removing incomplete records:", df.shape)

Shape after removing incomplete records: (5256, 12)


# Handle remaining Department Referral missing values

In [17]:
df["Department Referral"] = df["Department Referral"].fillna("No Referral")

# Handle Patient Satisfaction Score

In [18]:
median_satisfaction = df["Patient Satisfaction Score"].median()

print("Median satisfaction score:", median_satisfaction)

df["Patient Satisfaction Score"] = (
    df["Patient Satisfaction Score"]
    .fillna(median_satisfaction)
)

Median satisfaction score: 5.0


# Standardize Department names                  

In [19]:
df["Department Referral"] = (
    df["Department Referral"]
    .str.strip()
    .str.title()
)
print(df["Department Referral"].unique())

<StringArray>
['No Referral', 'General Practice', 'Orthopedics', 'Gastroenterology', 'Physiotherapy', 'Neurology', 'Cardiology', 'Renal']
Length: 8, dtype: str


# Check missing values again

In [20]:
missing_count = df.isnull().sum()

print(missing_count)

Patient Id                    0
Patient Admission Date        0
Patient First Initial         0
Patient Last Name             0
Patient Gender                0
Patient Age                   0
Patient Race                  0
Department Referral           0
Patient Admission Flag        0
Patient Satisfaction Score    0
Patient Waittime              0
Patient CM                    0
dtype: int64


In [21]:
total_missing = df.isnull().sum().sum()
total_cells = df.size

missing_percentage = (total_missing / total_cells) * 100

print("Total missing values:", total_missing)
print("Overall missing percentage:", round(missing_percentage, 2), "%")

Total missing values: 0
Overall missing percentage: 0.0 %


# Normalize healthcare indicators

In [22]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[[
    "Waittime Normalized",
    "Satisfaction Normalized"
]] = scaler.fit_transform(
    df[[
        "Patient Waittime",
        "Patient Satisfaction Score"
    ]]
)

In [23]:
print("Patient Waittime statistics:")
print(df["Patient Waittime"].describe())

print("\nPatient Satisfaction statistics:")
print(df["Patient Satisfaction Score"].describe())

Patient Waittime statistics:
count    5256.000000
mean       35.335236
std        14.840537
min        10.000000
25%        22.000000
50%        36.000000
75%        48.000000
max        60.000000
Name: Patient Waittime, dtype: float64

Patient Satisfaction statistics:
count    5256.000000
mean        4.996195
std         2.171344
min         0.000000
25%         5.000000
50%         5.000000
75%         5.000000
max        10.000000
Name: Patient Satisfaction Score, dtype: float64


In [24]:
print("Final dataset shape:", df.shape)

print("\nFinal missing values:")
print(df.isnull().sum())

print("\nFinal data types:")
print(df.dtypes)

Final dataset shape: (5256, 14)

Final missing values:
Patient Id                    0
Patient Admission Date        0
Patient First Initial         0
Patient Last Name             0
Patient Gender                0
Patient Age                   0
Patient Race                  0
Department Referral           0
Patient Admission Flag        0
Patient Satisfaction Score    0
Patient Waittime              0
Patient CM                    0
Waittime Normalized           0
Satisfaction Normalized       0
dtype: int64

Final data types:
Patient Id                               str
Patient Admission Date        datetime64[us]
Patient First Initial                    str
Patient Last Name                        str
Patient Gender                           str
Patient Age                            int64
Patient Race                             str
Department Referral                      str
Patient Admission Flag                  bool
Patient Satisfaction Score           float64
Patient Waittim

# Final dataset check

In [25]:
df.to_csv(
    "Hospital_ER_Cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
